In [8]:
import pandas as pd

print("Storage Telemetry Analysis Notebook Initialized")

Storage Telemetry Analysis Notebook Initialized


In [ ]:
# Load database config from YAML
import pandas as pd
from utils.notebook_helpers import get_postgres_engine

engine, pg_conf = get_postgres_engine("../configs/database.yaml")

raw_df = pd.read_sql_query("SELECT * FROM raw_device_metrics", engine)
curated_df = pd.read_sql_query("SELECT * FROM curated_device_metrics", engine)

raw_df["timestamp"] = pd.to_datetime(raw_df["timestamp"], errors="coerce")
curated_df["timestamp"] = pd.to_datetime(curated_df["timestamp"], errors="coerce")

print("Connected to PostgreSQL:", pg_conf["host"], pg_conf["db"])
print("raw shape:", raw_df.shape)
print("curated shape:", curated_df.shape)
print("date range:", raw_df["timestamp"].min(), "->", raw_df["timestamp"].max())

raw_df.head()

raw shape: (10080, 18)
curated shape: (10080, 40)
date range: 2026-03-30 00:01:58.457739+00:00 -> 2026-04-05 23:57:49.449651+00:00


,device,timestamp,r_s,w_s,rmb_s,wmb_s,r_await,w_await,aqu_sz,util_pct,rrqm_s,wrqm_s,rareq_sz,wareq_sz,svctm,iowait_pct,source_file,ingest_run_id
0,dm-0,2026-03-30 00:01:58.457739+00:00,130.543091,79.184408,9.858905,7.593411,0.515383,1.512394,1.228957,18.985405,6.652675,8.419268,76.046825,87.252764,0.375259,0.000000,data/raw/generated_iostat.csv,36445d12-a567-49cd-b36a-32fb1a6698ad
1,dm-0,2026-03-30 00:07:25.975778+00:00,140.798611,111.640416,13.975163,7.333685,0.722049,2.733421,0.337266,5.122736,6.204966,13.070514,54.499338,50.337116,0.314612,2.580240,data/raw/generated_iostat.csv,36445d12-a567-49cd-b36a-32fb1a6698ad
2,dm-0,2026-03-30 00:12:41.635044+00:00,125.286126,66.371400,12.555741,6.673198,0.747056,2.052067,0.148839,25.935530,4.115189,7.280113,73.759372,81.468518,0.483813,2.334648,data/raw/generated_iostat.csv,36445d12-a567-49cd-b36a-32fb1a6698ad
3,dm-0,2026-03-30 00:17:16.872031+00:00,100.356193,75.783214,8.572476,6.661890,0.920823,0.937210,0.292099,26.257811,8.018142,12.748040,71.675382,38.846390,0.434196,0.713070,data/raw/generated_iostat.csv,36445d12-a567-49cd-b36a-32fb1a6698ad
4,dm-0,2026-03-30 00:22:38.881089+00:00,108.479764,117.221000,7.843162,6.857079,1.402010,1.604284,0.796995,16.542202,12.173366,14.236511,86.284345,62.824981,0.349234,0.000000,data/raw/generated_iostat.csv,36445d12-a567-49cd-b36a-32fb1a6698ad


In [10]:
raw_profile_fields = [
    "r_s", "w_s", "rkB_s", "wkB_s", "await", "aqu_sz", "util",
    "rrqm_s", "wrqm_s", "rareq_sz", "wareq_sz", "svctm", "iowait_pct",
]
raw_profile_fields = [c for c in raw_profile_fields if c in raw_df.columns]
raw_df[raw_profile_fields].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T

,count,mean,std,min,50%,90%,95%,99%,max
r_s,10080.0,292.012684,317.663304,0.000000,214.354534,601.853463,769.745686,1246.630293,6658.959428
w_s,10080.0,216.338427,148.576107,9.593904,178.741863,413.107929,514.111224,688.383959,1937.805764
aqu_sz,10080.0,0.873684,1.622369,0.035151,0.496670,1.865614,2.721016,5.649435,92.694137
rrqm_s,10080.0,14.584096,12.578504,0.000000,10.894097,30.924729,39.540485,58.394372,155.359778
wrqm_s,10080.0,13.669898,13.860020,0.000000,9.132791,29.987001,41.992211,70.044015,110.533022
rareq_sz,10080.0,65.589121,43.729068,1.000000,58.956609,129.546753,155.102998,194.855159,262.773587
wareq_sz,10080.0,47.129424,34.574340,1.000000,41.245854,97.542242,117.145760,145.134322,213.852492
svctm,10080.0,7.665536,40.917075,0.080952,0.529179,10.729921,29.844901,147.198594,1159.652066
iowait_pct,10080.0,5.579863,7.419196,0.000000,3.014669,15.040076,21.919596,36.030119,70.250040


In [11]:
curated_profile_fields = [
    "total_iops",
    "avg_latency_ms",
    "util_pct",
    "aqu_sz",
    "saturation_score",
    "read_ratio",
    "write_ratio",
    "throughput_mb_s",
    "iops_total",
    "merge_rate_total",
    "merge_efficiency",
    "await_ratio",
    "svctm_await_ratio",
    "queue_efficiency",
    "write_amplification",
    "iowait_pressure",
]
curated_profile_fields = [c for c in curated_profile_fields if c in curated_df.columns]
curated_df[curated_profile_fields].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T

,count,mean,std,min,50%,90%,95%,99%,max
total_iops,10080.0,508.351111,428.642377,18.221891,400.387044,992.667256,1258.644263,1799.626111,7297.475831
avg_latency_ms,10080.0,52.568262,813.186919,0.196167,1.394085,33.218134,108.119421,773.454919,65239.601770
util_pct,10080.0,42.857076,28.274637,0.000000,37.241194,93.228534,100.000000,100.000000,100.000000
aqu_sz,10080.0,0.873684,1.622369,0.035151,0.496670,1.865614,2.721016,5.649435,92.694137
saturation_score,10080.0,49.768636,145.600399,0.000000,16.871256,113.058375,187.274976,483.910817,8789.858007
read_ratio,10080.0,0.538883,0.121434,0.000000,0.547107,0.684796,0.719579,0.789506,0.933809
write_ratio,10080.0,0.461117,0.121434,0.066191,0.452893,0.624006,0.680145,0.779224,1.000000
throughput_mb_s,10080.0,22.671970,22.035467,1.180264,16.145423,47.869226,64.199438,94.600674,358.967132
iops_total,10080.0,508.351111,428.642377,18.221891,400.387044,992.667256,1258.644263,1799.626111,7297.475831
merge_rate_total,10080.0,28.253994,24.691962,0.420885,20.477496,59.998254,79.685835,119.920934,169.745549


In [12]:
# Null and uniqueness checks for quality sanity
quality_profile = pd.DataFrame({
    "null_pct": (curated_df.isna().mean() * 100).round(2),
    "nunique": curated_df.nunique(),
}).sort_values("null_pct", ascending=False)
quality_profile.head(20)

,null_pct,nunique
device,0.0,5
timestamp,0.0,10080
r_s,0.0,10080
w_s,0.0,10080
rmb_s,0.0,10073
wmb_s,0.0,10079
r_await,0.0,10080
w_await,0.0,10078
aqu_sz,0.0,10080
util_pct,0.0,8991
